# Lotka–Volterra: a coupled nonlinear system

**Book:** §3.2 and Remark 3.1 &nbsp;·&nbsp; `ch03/lotka_volterra.ipynb`

$$\dot x = \alpha x - \beta x y,\qquad \dot y = \delta x y - \gamma y,$$

with $\alpha=2/3,\ \beta=4/3,\ \gamma=\delta=1,\ x(0)=y(0)=0.9$. No elementary closed form, so we
compare with a high-accuracy Runge–Kutta integration. The system conserves

$$V(x,y)=\delta x-\gamma\ln x+\beta y-\alpha\ln y,$$

which gives an *independent* check that never enters the loss.

**Formulation.** One network, two outputs. Both initial conditions hard:
$x=x_0+t\,\mathcal N_1$, $y=y_0+t\,\mathcal N_2$; input normalised as $t/T$. Loss is the sum of the
two residuals:

$$\mathcal{L}=\overline{(\dot x-\alpha x+\beta x y)^2}+\overline{(\dot y-\delta x y+\gamma y)^2}.$$

**Cell 2 is the honest part:** the same network over two periods *fails*. That failure is
causality violation, and it is repaired in Chapter 5 (`ch05/causality_violation.ipynb`).

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
AL, BE, GA, DE = 2/3, 4/3, 1.0, 1.0
X0 = Y0 = 0.9
rhs = lambda t, z: [AL*z[0]-BE*z[0]*z[1], DE*z[0]*z[1]-GA*z[1]]
invariant = lambda x, y: DE*x - GA*np.log(x) + BE*y - AL*np.log(y)

# One network, two outputs. Both initial conditions are hard, so the loss is just the sum of
# the two residuals. The long schedule is NOT optional: the predator spike is sharp, and a
# short run lands two orders of magnitude short -- try epochs=12000 and watch it degrade.
def solve_pinn(T, epochs=30000, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(),
                        nn.Linear(64,64), nn.Tanh(), nn.Linear(64,2)).to(device)
    def xy(t):
        o = net(t/T)                                  # normalised input, O(1)
        return X0 + t*o[:,0:1], Y0 + t*o[:,1:2]       # both ICs exact
    opt = torch.optim.Adam(net.parameters(), 3e-3)
    t0 = time.perf_counter()
    for e in range(epochs):
        if e == int(0.50*epochs):
            for g in opt.param_groups: g['lr'] = 6e-4
        if e == int(0.75*epochs):
            for g in opt.param_groups: g['lr'] = 1.2e-4
        if e == int(0.90*epochs):
            for g in opt.param_groups: g['lr'] = 3e-5
        opt.zero_grad()
        t = (torch.rand(1024,1,device=device)*T).requires_grad_(True)
        x, y = xy(t)
        rx = g1(x,t) - (AL*x - BE*x*y)
        ry = g1(y,t) - (DE*x*y - GA*y)
        ((rx**2).mean() + (ry**2).mean()).backward(); opt.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    return xy, time.perf_counter()-t0

T = 6.0                                                # a little under one period
xy, dt_train = solve_pinn(T)
print(f'training: {dt_train:.0f} s')

ref = solve_ivp(rhs, [0,T], [X0,Y0], rtol=1e-10, atol=1e-12, dense_output=True)
tg  = np.linspace(0, T, 600)
xe, ye = ref.sol(tg)
tt = torch.tensor(tg, dtype=torch.float32, device=device).reshape(-1,1)
with torch.no_grad(): xp, yp = xy(tt)
xp = xp.cpu().numpy().ravel(); yp = yp.cpu().numpy().ravel()
ex = np.sqrt(np.mean((xp-xe)**2)/np.mean(xe**2))
ey = np.sqrt(np.mean((yp-ye)**2)/np.mean(ye**2))
V  = invariant(xp, yp); drift = abs(V.max()-V.min())/abs(V.mean())
print(f'rel L2:  x = {ex:.1e}   y = {ey:.1e}')
print(f'invariant drift = {drift:.1e}   -- conserved, though never imposed')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(tg, xe, 'g',  lw=2.6, alpha=.55, label='prey $x$ (RK)')
ax[0].plot(tg, ye, 'b',  lw=2.6, alpha=.55, label='predator $y$ (RK)')
ax[0].plot(tg, xp, 'r--', lw=1.5, label=f'PINN $x$ ({ex:.1e})')
ax[0].plot(tg, yp, 'k--', lw=1.5, label=f'PINN $y$ ({ey:.1e})')
ax[0].set_xlabel('t'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[0].set_title('Populations over $t\\in[0,6]$')
ax[1].plot(tg, V, 'r', lw=1.4)
ax[1].axhline(invariant(X0, Y0), color='g', lw=2.4, alpha=.5, label='exact invariant')
ax[1].set_xlabel('t'); ax[1].set_ylabel('$V(x,y)$'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
ax[1].set_title(f'Conserved quantity, never imposed (drift {drift:.0e})')
plt.tight_layout(); plt.show()

In [ ]:
# The honest limitation (Remark 3.1): stretch the horizon to two periods and the SAME
# network, trained identically, falls apart. This is causality violation -- see ch05.
T2 = 15.0
xy2, _ = solve_pinn(T2)
ref2 = solve_ivp(rhs, [0,T2], [X0,Y0], rtol=1e-10, atol=1e-12, dense_output=True)
tg2 = np.linspace(0, T2, 1200); xe2, ye2 = ref2.sol(tg2)
tt2 = torch.tensor(tg2, dtype=torch.float32, device=device).reshape(-1,1)
with torch.no_grad(): xp2, yp2 = xy2(tt2)
xp2 = xp2.cpu().numpy().ravel(); yp2 = yp2.cpu().numpy().ravel()
ex2 = np.sqrt(np.mean((xp2-xe2)**2)/np.mean(xe2**2))
print(f'rel L2 in x over two periods: {ex2:.2f}   <-- FAILURE (was {ex:.1e} over one)')

plt.figure(figsize=(9,4))
plt.plot(tg2, xe2, 'g', lw=2.6, alpha=.55, label='prey $x$ (RK)')
plt.plot(tg2, xp2, 'r--', lw=1.5, label=f'PINN $x$ (rel $L_2$ = {ex2:.2f})')
plt.xlabel('t'); plt.legend(); plt.grid(alpha=.3)
plt.title('Two periods: the PINN loses the oscillation entirely')
plt.tight_layout(); plt.show()